# Notebook 14 — Measurement-aware and selection-aware CYP regression

**Scientific question:** Are CYP potency predictions improved by (a) respecting Bayesian dose-response confidence intervals as *intervals*, and (b) acknowledging the nonrandom selection of compounds for quantitative measurement?

**Five matched experimental arms:** standard point LightGBM; smooth point-Huber LightGBM; mixed interval/point Huber; propensity-weighted point Huber; propensity-weighted interval Huber.

**Selection model:** among compounds appearing in the primary *single-concentration screen*, predict whether an isoform's **direct-inhibition pIC50 was measured**, using **structure-only features**. Weights for labeled training compounds are estimated via inner scaffold cross-fitting and are clipped. Missing pIC50 is **never labeled zero/inactive**. The selection model is exploratory: observational missingness can remain nonignorable (MNAR), and weighting does not establish causal identification.

**Validation:** five Bemis–Murcko outer folds for the quantitative endpoints; screening-only auxiliary structures from held-out scaffolds are excluded from each training propensity model. Optional descriptive observed-screen-response audit does not enter inference.

References: https://huggingface.co/datasets/openadmet/cyp-challenge-train-test ; https://github.com/OpenADMET/CYP-Challenge-Tutorial

## 0 · Colab packages (CPU; GPU not required)

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.5" "scipy>=1.11" pandas matplotlib

## 1 · Configuration, sources and settings

Use `FAST_MODE=True` for a one-fold smoke test, then `False` for five outer folds. Set `PERSISTENT_ROOT` for Google Drive restartability. Alter `EXPERIMENT_TAG` if you change the protocol.

In [ ]:
import os, io, json, zipfile, hashlib, platform, warnings
from pathlib import Path
from urllib.request import urlopen
import numpy as np, pandas as pd, lightgbm as lgb
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, brier_score_loss
import matplotlib.pyplot as plt

ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
SEED=2026
FAST_MODE=True
OUTER_FOLDS=5
FOLDS_TO_RUN=[0] if FAST_MODE else list(range(OUTER_FOLDS))
PROPENSITY_INNER_FOLDS=2 if FAST_MODE else 3
FP_BITS=512
N_ESTIMATORS=65 if FAST_MODE else 300
PROP_ESTIMATORS=45 if FAST_MODE else 140
INTERVAL_WEIGHT=0.80  # smooth interval objective retains a point-anchor component to avoid flat loss
HUBER_DELTA=0.50
PROPENSITY_FLOOR=0.10
PROPENSITY_CAP=0.95
MAX_IPW=3.0
IPW_POWER=0.75
EXPERIMENT_TAG='measurement_selection_v1'
DATA_OVERRIDES={}  # {'direct':'/content/direct.csv','screen':'/content/screen.csv'}
PERSISTENT_ROOT=None  # '/content/drive/MyDrive/CYP/Notebook14'
ROOT_URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'
FILENAMES={'direct':'cyp-challenge-TRAIN_inhibition.csv',
           'screen':'cyp-challenge-single-concentration-TRAIN.csv'}
BASE=Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
OUT=BASE/'cyp_notebook_14_results';OUT.mkdir(parents=True,exist_ok=True)
CACHE=BASE/'cyp_notebook_14_cache';CACHE.mkdir(parents=True,exist_ok=True)
np.random.seed(SEED)
print('RDKit',rdBase.rdkitVersion,'LightGBM',lgb.__version__,'FAST_MODE',FAST_MODE)

def read_source(which):
    blob=Path(DATA_OVERRIDES[which]).read_bytes() if which in DATA_OVERRIDES else urlopen(ROOT_URL+FILENAMES[which],timeout=180).read()
    return pd.read_csv(io.BytesIO(blob)),hashlib.sha256(blob).hexdigest()

def ident(s):
    m=Chem.MolFromSmiles(str(s)) if pd.notna(s) else None
    if m is None:return None,None
    cs=Chem.MolToSmiles(m,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(m)
    sc=Chem.MolToSmiles(core,canonical=True,isomericSmiles=False) if core.GetNumAtoms() else 'ACYCLIC_'+cs
    return cs,sc

def interval_excess(pred,lo,hi):
    return np.maximum(np.asarray(lo)-pred,0.)+np.maximum(pred-np.asarray(hi),0.)

def st_rae(y,p,lo,hi):
    y,p,lo,hi=map(lambda x:np.asarray(x,float),(y,p,lo,hi))
    assert len(y)>=2 and np.all(np.isfinite(y)) and np.all(np.isfinite(lo)) and np.all(np.isfinite(hi))
    den=np.sum(interval_excess(np.full_like(y,y.mean()),lo,hi))
    return float(np.sum(interval_excess(p,lo,hi))/den) if den>1e-12 else np.nan

def export_zip():
    target=BASE/'cyp_notebook_14_results.zip'
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob('*')):
            if p.is_file():z.write(p,OUT.name+'/'+str(p.relative_to(OUT)))
    return target

## 2 · Build the measured and screened populations

The direct table supplies quantitative pIC50 and the experimental 95% credible interval, when available. The single-concentration file supplies the **screened population** (`enzyme`, `log2fc_estimate`). For a screened compound–CYP pair, the selection target is `1` if a quantitative direct pIC50 appears in the public direct table and `0` otherwise. This is an observation indicator, **not** an inhibition label.

In [ ]:
direct,sha_direct=read_source('direct')
screen_raw,sha_screen=read_source('screen')
assert {'SMILES','enzyme','log2fc_estimate'}.issubset(screen_raw.columns)
assert {'SMILES'}|{f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS} <= set(direct.columns)
primary=direct.copy().reset_index(drop=True)
primary_ids=[ident(s) for s in primary.SMILES]
primary['canonical_smiles']=[k for k,g in primary_ids]
primary['scaffold_group']=[g for k,g in primary_ids]
assert primary.canonical_smiles.notna().all() and primary.canonical_smiles.is_unique
N=len(primary)
primary_scaffolds=primary.scaffold_group.to_numpy(str)
Y=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition'],errors='coerce').to_numpy(float) for iso in ISOFORMS])
LOW=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_low',pd.Series(np.nan,index=primary.index)),errors='coerce').to_numpy(float) for iso in ISOFORMS])
HIGH=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_high',pd.Series(np.nan,index=primary.index)),errors='coerce').to_numpy(float) for iso in ISOFORMS])
valid_bounds=(np.isfinite(LOW)&np.isfinite(HIGH))
if np.any(valid_bounds & (LOW>HIGH)):
    raise ValueError('Inverted experimental confidence intervals found. Check source data.')
for j in range(4):
    missing_bounds=np.isfinite(Y[:,j])&~valid_bounds[:,j]
    # Missing CI is not fabricated: these records contribute only the point part of the objective.
    print(ISOFORMS[j],'quantitative labels',int(np.isfinite(Y[:,j]).sum()),'valid confidence intervals',int((np.isfinite(Y[:,j])&valid_bounds[:,j]).sum()))

folds=np.full(N,-1,int)
for f,(_,te) in enumerate(GroupKFold(n_splits=OUTER_FOLDS).split(np.arange(N),groups=primary_scaffolds)):
    folds[te]=f
assert not pd.DataFrame({'scaffold':primary_scaffolds,'fold':folds}).groupby('scaffold').fold.nunique().gt(1).any()
primary['fold']=folds
primary[['canonical_smiles','scaffold_group','fold']].to_csv(OUT/'fold_manifest.csv',index=False,float_format='%.12g')

screen=screen_raw.copy()
screen['canonical_smiles']=[ident(s)[0] for s in screen.SMILES]
screen['enzyme']=screen.enzyme.astype(str).str.strip().str.upper()
screen['log2fc_estimate']=pd.to_numeric(screen.log2fc_estimate,errors='coerce')
screen=screen.loc[screen.canonical_smiles.notna() & screen.enzyme.isin(ISOFORMS) & screen.log2fc_estimate.notna()].copy()
screen=screen.groupby(['canonical_smiles','enzyme'],as_index=False).agg(
    measured_screen_log2fc=('log2fc_estimate','mean'),screen_replicates=('log2fc_estimate','size'))
screen_wide=screen.pivot(index='canonical_smiles',columns='enzyme',values='measured_screen_log2fc')
for iso in ISOFORMS:
    assert iso in screen_wide.columns, f'No primary screening rows for {iso}'
extra=sorted(set(screen_wide.index)-set(primary.canonical_smiles))
ALL_SMILES=primary.canonical_smiles.tolist()+extra
ALL_GROUPS=np.asarray([ident(s)[1] for s in ALL_SMILES],object)
INDEX={s:i for i,s in enumerate(ALL_SMILES)}
NN=len(ALL_SMILES)
SC=np.full((NN,4),np.nan,float)
for s,r in screen_wide.iterrows():
    SC[INDEX[s]]=[r.get(iso,np.nan) for iso in ISOFORMS]
# A screened compound with an observed direct pIC50 is 'selected' for a quantitative DRC label.
SELECT=np.full((NN,4),np.nan)
for j in range(4):SELECT[np.isfinite(SC[:,j]),j]=0.
for i,s in enumerate(primary.canonical_smiles):
    for j in range(4):
        if np.isfinite(SC[i,j]) and np.isfinite(Y[i,j]):SELECT[i,j]=1.
assert np.isin(SELECT[np.isfinite(SELECT)],[0.,1.]).all()
print('Direct molecules',N,'screened molecules',len(screen_wide),'screen-only molecules',len(extra))
print('screen observed counts',np.isfinite(SC).sum(axis=0),'positive DRC observation counts',np.nansum(SELECT,axis=0))

## 3 · Structural descriptors for measured and screen-only molecules

All features are structure-derived and therefore also available at inference. **Observed log2FC screening measurements are not used as potency-model inputs**, avoiding reliance on unavailable blind-test assays.

In [ ]:
MOLS=[Chem.MolFromSmiles(s) for s in ALL_SMILES]
DESC_NAMES=['MolWt','MolLogP','TPSA','NumHDonors','NumHAcceptors','NumRotatableBonds',
    'RingCount','NumAromaticRings','FractionCSP3','HeavyAtomCount','NumHeteroatoms',
    'NHOHCount','NOCount','NumAliphaticRings','NumSaturatedRings','LabuteASA']
DESC=np.array([[getattr(Descriptors,name)(m) for name in DESC_NAMES] for m in MOLS],np.float32)
DESC[~np.isfinite(DESC)]=np.nan
fp_gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=FP_BITS)
FP=np.zeros((NN,FP_BITS),np.float32)
for i,m in enumerate(MOLS):DataStructs.ConvertToNumpyArray(fp_gen.GetCountFingerprint(m),FP[i])
FEATURES=np.column_stack([DESC,FP])
COLS=[f'd_{name}' for name in DESC_NAMES]+[f'm_{i}' for i in range(FP_BITS)]
def mat(rows):return pd.DataFrame(FEATURES[np.asarray(rows,int)],columns=COLS)
print('Feature matrix',FEATURES.shape)

## 4 · Interval pseudo-Huber objective and propensity weights

The custom gradient combines smooth interval distance and smooth point error. Inside a *valid* observed CI, the interval term has zero gradient; the point term still anchors learning. Rows without CI receive point-only gradient/hessian, **not fabricated bounds**.

The inverse propensity weights are **cross-fitted** using the screened population of the outer-training scaffolds. Scores are clipped to avoid extreme importance weights. Weights are not guaranteed to remove MNAR bias; compare their performance to the unweighted controls.

In [ ]:
def ph_grad_hess(err,delta=HUBER_DELTA):
    zz=np.square(np.asarray(err)/delta)
    return np.asarray(err)/np.sqrt(1.+zz), np.power(1.+zz,-1.5)

def objective_factory(y,lower,upper,weights=None,interval_mix=0.):
    y=np.asarray(y,float);lower=np.asarray(lower,float);upper=np.asarray(upper,float)
    ok=np.isfinite(lower)&np.isfinite(upper)&(lower<=upper)
    mix=np.where(ok,float(interval_mix),0.).astype(float)
    lo=np.where(ok,lower,y);hi=np.where(ok,upper,y)
    w=np.ones(len(y),float) if weights is None else np.asarray(weights,float)
    assert np.isfinite(y).all() and np.isfinite(w).all() and np.all(w>0)
    w=w/np.mean(w)
    def obj(y_true,y_pred):
        # This LightGBM custom objective is built for exactly the training rows passed to .fit.
        if len(y_pred)!=len(y):raise RuntimeError('Objective and training data rows became misaligned')
        pg,ph=ph_grad_hess(np.asarray(y_pred)-y)
        ie=np.where(np.asarray(y_pred)<lo,np.asarray(y_pred)-lo,
                    np.where(np.asarray(y_pred)>hi,np.asarray(y_pred)-hi,0.))
        ig,ih=ph_grad_hess(ie)
        ig[ie==0]=0.  # no interval gradient inside CI
        gradient=((1-mix)*pg+mix*ig)*w
        hessian=((1-mix)*ph+mix*ih)*w
        return gradient,np.maximum(hessian,1e-6)
    return obj

# Self-tests: gradients inside CI depend only on point anchor; outside push towards interval.
g,h=objective_factory([5.],[4.8],[5.2],interval_mix=.8)(np.array([5.]),np.array([5.]))
assert abs(float(g[0]))<1e-12 and float(h[0])>0
g,_=objective_factory([5.],[4.8],[5.2],interval_mix=.8)(np.array([5.]),np.array([4.]))
assert g[0]<0

def make_regressor(seed,objective='regression'):
    return lgb.LGBMRegressor(objective=objective,n_estimators=N_ESTIMATORS,learning_rate=.045,
        num_leaves=15,min_child_samples=8,reg_lambda=5.,verbosity=-1,n_jobs=2,random_state=int(seed))

def make_selection_model(seed):
    return lgb.LGBMClassifier(n_estimators=PROP_ESTIMATORS,learning_rate=.045,
        num_leaves=9,min_child_samples=12,reg_lambda=8.,verbosity=-1,n_jobs=2,random_state=int(seed))

def predict_propensity(train_rows,predict_rows,j,seed):
    vals=SELECT[np.asarray(train_rows,int),j]
    assert np.isfinite(vals).all()
    if len(np.unique(vals))<2:
        return np.full(len(predict_rows),float(np.mean(vals)) if len(vals) else .5)
    mdl=make_selection_model(seed)
    mdl.fit(mat(train_rows),vals.astype(int))
    return mdl.predict_proba(mat(predict_rows))[:,1]

def crossfit_propensity(outer_train,j,seed):
    # Train selection model only on screened molecules outside outer validation scaffolds.
    train_groups=set(primary_scaffolds[outer_train])
    holdout_groups=set(primary_scaffolds)-train_groups
    candidates=np.flatnonzero(np.isfinite(SELECT[:,j]) & ~np.isin(ALL_GROUPS,list(holdout_groups)))
    # Isolate even screen-only structures whose scaffolds are in the held-out set.
    assert not (set(ALL_GROUPS[candidates])&holdout_groups)
    if len(candidates)<4:return {},None,{'screened_train':len(candidates),'crossfit_groups':0}
    fold_groups=np.asarray(ALL_GROUPS[candidates],str)
    splits=min(PROPENSITY_INNER_FOLDS,len(set(fold_groups)))
    pred=np.full(len(candidates),np.nan,float)
    if splits>=2:
        for k,(tr,te) in enumerate(GroupKFold(n_splits=splits).split(candidates,groups=fold_groups)):
            assert not (set(fold_groups[tr]) & set(fold_groups[te]))
            pred[te]=predict_propensity(candidates[tr],candidates[te],j,seed+100+k)
    else:
        pred[:]=np.nanmean(SELECT[candidates,j])
    assert np.isfinite(pred).all()
    # True propensity is unknown; this is only a cross-fitted predictive estimate.
    crossfit=dict(zip(candidates.tolist(),pred.tolist()))
    # Generalization of selection prediction to outer validation screened compounds.
    valid_screen=np.flatnonzero(np.isfinite(SELECT[:,j]) & np.isin(ALL_GROUPS,list(holdout_groups)))
    if len(valid_screen):
        hold_pred=predict_propensity(candidates,valid_screen,j,seed+777)
        truth=SELECT[valid_screen,j]
        auc=roc_auc_score(truth,hold_pred) if len(set(truth))==2 else np.nan
        brier=brier_score_loss(truth,hold_pred)
        evaluation={'screened_train':len(candidates),'screened_holdout':len(valid_screen),
           'holdout_auc':auc,'holdout_brier':brier,'crossfit_groups':splits}
    else:evaluation={'screened_train':len(candidates),'screened_holdout':0,
        'holdout_auc':np.nan,'holdout_brier':np.nan,'crossfit_groups':splits}
    return crossfit,pred,evaluation

## 5 · Fit five model variants on identical outer folds

Each model sees the same labeled, outer-training structures. Selection-status training is restricted to outer-training scaffolds (including screen-only compounds where eligible). The IPW variant computes weights only for **measured pIC50 compounds that were also in the screen**; other measured compounds retain unit weight. `N_ESTIMATORS` and all model hyperparameters are held fixed.

In [ ]:
MODELS=['Point_LGBM','Point_Huber','Interval_Huber','IPW_Point_Huber','IPW_Interval_Huber']
SETTINGS=dict(tag=EXPERIMENT_TAG,sha_direct=sha_direct,sha_screen=sha_screen,seed=SEED,
    fast=FAST_MODE,fp_bits=FP_BITS,n_estimators=N_ESTIMATORS,prop_estimators=PROP_ESTIMATORS,
    prop_inner=PROPENSITY_INNER_FOLDS,interval_weight=INTERVAL_WEIGHT,huber_delta=HUBER_DELTA,
    propensity_floor=PROPENSITY_FLOOR,propensity_cap=PROPENSITY_CAP,max_ipw=MAX_IPW,
    ipw_power=IPW_POWER,lgb_version=lgb.__version__,rdkit=rdBase.rdkitVersion)
RUN_ID=hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()[:16]
RUN_CACHE=CACHE/RUN_ID;RUN_CACHE.mkdir(parents=True,exist_ok=True)

PRED_FRAMES=[];AUDIT_FRAMES=[];PROP_FRAMES=[]
for fold in FOLDS_TO_RUN:
    pred_path=RUN_CACHE/f'pred_fold_{fold}.csv'
    audit_path=RUN_CACHE/f'audit_fold_{fold}.csv'
    props_path=RUN_CACHE/f'propensity_fold_{fold}.csv'
    tr=np.flatnonzero(folds!=fold);va=np.flatnonzero(folds==fold)
    assert not set(primary_scaffolds[tr])&set(primary_scaffolds[va])
    expected={(int(i),iso) for i in va for j,iso in enumerate(ISOFORMS) if np.isfinite(Y[i,j])}
    if pred_path.exists() and audit_path.exists() and props_path.exists():
        p=pd.read_csv(pred_path);a=pd.read_csv(audit_path);pr=pd.read_csv(props_path)
        assert {(int(r),str(iso)) for r,iso in zip(p.row_index,p.isoform)}==expected
        print('RESUMED fold',fold,'measurements',len(p))
        PRED_FRAMES.append(p);AUDIT_FRAMES.append(a);PROP_FRAMES.append(pr)
        continue
    pred_rows=[];audit_rows=[];prop_rows=[]
    for j,iso in enumerate(ISOFORMS):
        labeled=tr[np.isfinite(Y[tr,j])]
        val=va[np.isfinite(Y[va,j])]
        if len(labeled)<10 or not len(val):continue
        status_pred,_,prop_diagnostics=crossfit_propensity(tr,j,SEED+71*fold+5*j)
        weights=np.ones(len(labeled),float)
        for i,k in enumerate(labeled):
            if k in status_pred: # measured direct pIC50 and screened
                p=float(np.clip(status_pred[k],PROPENSITY_FLOOR,PROPENSITY_CAP))
                weights[i]=min(MAX_IPW,p**(-IPW_POWER))
        weights/=np.mean(weights)
        lower=LOW[labeled,j];upper=HIGH[labeled,j];targets=Y[labeled,j]
        xtrain=mat(labeled);xval=mat(val)
        base=make_regressor(SEED+fold*17+j)
        base.fit(xtrain,targets)
        model_preds={'Point_LGBM':base.predict(xval)}
        configs=[('Point_Huber',0.,None),('Interval_Huber',INTERVAL_WEIGHT,None),
                 ('IPW_Point_Huber',0.,weights),('IPW_Interval_Huber',INTERVAL_WEIGHT,weights)]
        # Custom LightGBM objectives initialize their raw score at 0 rather than y.mean.
        # Center outcomes AND CI bounds on the training median to avoid saturated Huber
        # gradients at pIC50 ~ 4–8; add back the offset at prediction time.
        label_offset=float(np.median(targets))
        for ii,(name,mix,w) in enumerate(configs):
            obj=objective_factory(targets-label_offset,lower-label_offset,upper-label_offset,weights=w,interval_mix=mix)
            reg=make_regressor(SEED+fold*17+j,objective=obj)
            reg.fit(xtrain,targets-label_offset)
            model_preds[name]=reg.predict(xval)+label_offset
        for k,row in enumerate(val):
            rec=dict(row_index=int(row),fold=int(fold),isoform=iso,
                canonical_smiles=primary.canonical_smiles.iloc[row],scaffold_group=primary_scaffolds[row],
                y_true=float(Y[row,j]),ci_low=float(LOW[row,j]),ci_high=float(HIGH[row,j]))
            rec.update({name:float(arr[k]) for name,arr in model_preds.items()})
            pred_rows.append(rec)
        audit_rows.append(dict(fold=fold,isoform=iso,outer_scaffold_overlap=0,
             n_direct_train=len(labeled),n_direct_val=len(val),
             n_valid_train_ci=int(np.sum(np.isfinite(lower)&np.isfinite(upper))),
             n_propensity_weighted=int(np.sum([k in status_pred for k in labeled])),
             ipw_min=float(weights.min()),ipw_max=float(weights.max()),**prop_diagnostics))
        prop_rows.append(dict(fold=fold,isoform=iso,selected_screened_train=prop_diagnostics['screened_train'],
            screened_holdout=prop_diagnostics.get('screened_holdout',0),
            selection_holdout_auc=prop_diagnostics.get('holdout_auc',np.nan),
            selection_holdout_brier=prop_diagnostics.get('holdout_brier',np.nan)))
    p=pd.DataFrame(pred_rows);a=pd.DataFrame(audit_rows);pr=pd.DataFrame(prop_rows)
    if not p.empty:assert {(int(r),str(iso)) for r,iso in zip(p.row_index,p.isoform)}==expected
    p.to_csv(pred_path,index=False,float_format='%.12g');a.to_csv(audit_path,index=False,float_format='%.12g');pr.to_csv(props_path,index=False,float_format='%.12g')
    PRED_FRAMES.append(p);AUDIT_FRAMES.append(a);PROP_FRAMES.append(pr)
    print('FIT fold',fold,'observations',len(p),'selection records',a.screened_train.sum() if len(a) else 0)
PRED=pd.concat(PRED_FRAMES,ignore_index=True)
AUDIT=pd.concat(AUDIT_FRAMES,ignore_index=True)
PROP_DF=pd.concat(PROP_FRAMES,ignore_index=True)
assert (AUDIT.outer_scaffold_overlap==0).all()
PRED.to_csv(OUT/'validation_predictions.csv',index=False,float_format='%.12g')
# Use the exact serialized rows for all downstream diagnostics on both fresh and resumed runs.
# This prevents harmless float parse/roundtrip changes in derived CSV hashes.
PRED=pd.read_csv(OUT/'validation_predictions.csv')
AUDIT.to_csv(OUT/'training_and_leakage_audit.csv',index=False,float_format='%.12g')
PROP_DF.to_csv(OUT/'selection_model_diagnostics.csv',index=False,float_format='%.12g')

## 6 · Compare point, interval-aware and selection-weighted variants

Report pIC50 MAE and challenge-style ST-RAE **separately** (ST-RAE only where credible bounds exist). These are internal validation estimates, not blind-test scores. ST-RAE denominators are calculated across the concatenated held-out predictions for each CYP, not averaged over tiny folds.

In [ ]:
metrics=[]
for iso,part in PRED.groupby('isoform'):
    y=part.y_true.to_numpy(float)
    m=part[['ci_low','ci_high']].notna().all(axis=1).to_numpy()
    for model in MODELS:
        pred=part[model].to_numpy(float)
        row=dict(isoform=iso,model=model,n=len(part),mae=mean_absolute_error(y,pred),
            r2=r2_score(y,pred),bias=float(np.mean(pred-y)),
            pred_std_ratio=float(np.std(pred)/(np.std(y)+1e-12)),n_ci=int(m.sum()))
        row['mean_interval_excess']=float(np.mean(interval_excess(pred[m],part.ci_low[m],part.ci_high[m]))) if m.any() else np.nan
        row['st_rae']=st_rae(y[m],pred[m],part.ci_low[m],part.ci_high[m]) if m.sum()>1 else np.nan
        metrics.append(row)
METRICS=pd.DataFrame(metrics)
METRICS.to_csv(OUT/'per_isoform_metrics.csv',index=False,float_format='%.12g')
macro=METRICS.groupby('model',as_index=False).agg(macro_mae=('mae','mean'),
    macro_st_rae=('st_rae','mean'),macro_interval_excess=('mean_interval_excess','mean'))
macro.to_csv(OUT/'macro_summary.csv',index=False,float_format='%.12g')
print(macro.sort_values('macro_st_rae').to_string(index=False))

# Compare weak/high potency ranges and wide/narrow CI bins using *observed labels* for post hoc diagnostics only.
strata=[]
for iso,part in PRED.groupby('isoform'):
    y=part.y_true.to_numpy(float)
    lo,hi=np.nanquantile(y,[.1,.9])
    interval_width=part.ci_high-part.ci_low
    width_median=np.nanmedian(interval_width) if np.isfinite(interval_width).any() else np.nan
    for stratum,mask in [('lowest_pIC50_decile',y<=lo),('highest_pIC50_decile',y>=hi),
        ('wide_CI',np.asarray(interval_width>width_median)),('narrow_CI',np.asarray(interval_width<=width_median))]:
        if mask.sum()==0:continue
        for model in MODELS:
            strata.append(dict(isoform=iso,stratum=stratum,model=model,n=int(mask.sum()),
               mae=float(np.mean(np.abs(y[mask]-part[model].to_numpy(float)[mask])))))
STRATA=pd.DataFrame(strata)
STRATA.to_csv(OUT/'potency_and_ci_stratified_errors.csv',index=False,float_format='%.12g')

fig,axs=plt.subplots(1,2,figsize=(11,4))
for _,r in macro.iterrows():
    axs[0].bar(r.model,r.macro_mae);axs[1].bar(r.model,r.macro_st_rae)
for ax,title in zip(axs,['Macro MAE','Macro ST-RAE']):
    ax.set_title(title);ax.tick_params(axis='x',rotation=65)
fig.tight_layout();fig.savefig(OUT/'measurement_aware_model_comparison.png',dpi=130);plt.show()

## 7 · Screening selection diagnostics (descriptive only)

Compare selection probability by **measured primary screening response**, but do not use that response as a blind-inference feature. The original screening assay is under the active-preincubation condition and negative log2FC indicates inhibition; observed values are not equivalent to pIC50.

In [ ]:
selection_records=[]
for j,iso in enumerate(ISOFORMS):
    idx=np.flatnonzero(np.isfinite(SC[:,j]) & np.isfinite(SELECT[:,j]))
    if not len(idx):continue
    for i in idx:
        selection_records.append(dict(isoform=iso,screen_log2fc=float(SC[i,j]),
            obtained_direct_label=int(SELECT[i,j]),is_direct_training_compound=int(i<N)))
sel_df=pd.DataFrame(selection_records)
sel_df['screen_response_bin']=pd.qcut(sel_df.screen_log2fc,q=5,duplicates='drop')
sel_agg=sel_df.groupby(['isoform','screen_response_bin'],observed=True).agg(
    n=('obtained_direct_label','size'),observed_fraction=('obtained_direct_label','mean')).reset_index()
sel_agg.to_csv(OUT/'empirical_selection_by_screen_response.csv',index=False,float_format='%.12g')
print('Selection fraction among screened compound–CYP pairs:')
print(sel_df.groupby('isoform').obtained_direct_label.agg(['size','mean']).to_string())

## 8 · Export and scientific cautions

- A flat interval loss has no incentive to distinguish pIC50 values within the same credible region; the point-anchor component is intentional.
- **Missing measured pIC50 is not known inactivity.** The selection indicator is a label-observation proxy, and the screening dataset may itself omit a prior selection stage.
- IPW requires an untestable missing-at-random assumption conditional on the variables; a sensitivity analysis is warranted before interpreting corrected potency as unbiased.
- The experiment cannot estimate blind-test MA-ST-RAE without withheld outcomes. Never rank submission candidates by blind test feature distributions.
- The model families share fixed hyperparameters. Retrospective selection of the best arm on five-fold OOF still needs independent confirmation.

In [ ]:
metadata={'notebook':'14_interval_and_selection_aware_regression.ipynb',
    'run_id':RUN_ID,'settings':SETTINGS,'folds':FOLDS_TO_RUN,
    'complete_cv':set(FOLDS_TO_RUN)==set(range(OUTER_FOLDS)),
    'n_direct_molecules':N,'n_screened_molecules':len(screen_wide),
    'n_screen_only_molecules':len(extra),'source_sha256':{'direct':sha_direct,'screen':sha_screen},
    'evaluation':'PUBLIC_TRAIN_GROUPED_CV_NOT_BLIND_TEST',
    'python':platform.python_version()}
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
archive=export_zip()
print('Saved:',archive.resolve())
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass